In [ ]:
# %%writefile 02_resume_training.ipynb
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Resume Training\n",
    "# Pick up exactly where you left off (Colab disconnected? No problem!)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# LOAD RESUME HELPER\n",
    "%run resume_helper.ipynb\n",
    "\n",
    "# Check state\n",
    "state = load_state()\n",
    "if not state:\n",
    "    print(\"❌ No saved state. Run 01_train.ipynb first!\")\n",
    "    raise SystemExit\n",
    "\n",
    "CHECKPOINT_DIR = find_latest_checkpoint(state['output_dir'])\n",
    "if not CHECKPOINT_DIR:\n",
    "    print(\"❌ No checkpoints found!\")\n",
    "    raise SystemExit\n",
    "\n",
    "print(f\"✅ Will resume from: {CHECKPOINT_DIR}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# RE-INSTALL (Colab fresh session)\n",
    "!pip install -q transformers datasets accelerate peft bitsandbytes trl"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# IMPORTS (same as original)\n",
    "import torch\n",
    "from transformers import (\n",
    "    AutoModelForCausalLM,\n",
    "    AutoTokenizer,\n",
    "    TrainingArguments,\n",
    "    BitsAndBytesConfig\n",
    ")\n",
    "from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel\n",
    "from trl import SFTTrainer\n",
    "from datasets import load_dataset, concatenate_datasets\n",
    "\n",
    "print(\"✅ Imports ready\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# LOAD FROM CHECKPOINT (not from scratch!)\n",
    "print(\"🔄 Loading model from checkpoint...\")\n",
    "\n",
    "# Configs (same as before)\n",
    "bnb_config = BitsAndBytesConfig(\n",
    "    load_in_4bit=True,\n",
    "    bnb_4bit_quant_type=\"nf4\",\n",
    "    bnb_4bit_compute_dtype=torch.float16,\n",
    "    bnb_4bit_use_double_quant=True,\n",
    ")\n",
    "\n",
    "tokenizer = AutoTokenizer.from_pretrained(state['output_dir'])\n",
    "\n",
    "# Load base model\n",
    "base_model = AutoModelForCausalLM.from_pretrained(\n",
    "    \"codellama/CodeLlama-7b-Instruct-hf\",\n",
    "    quantization_config=bnb_config,\n",
    "    device_map=\"auto\",\n",
    ")\n",
    "\n",
    "# Load LoRA adapter from checkpoint\n",
    "model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR)\n",
    "\n",
    "print(f\"✅ Model loaded from step {state['last_step']}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# RE-CREATE DATASET (or load from saved)\n",
    "print(\"📚 Re-creating dataset...\")\n",
    "\n",
    "code_dataset = load_dataset(\"iamtarun/python_code_instructions_18k_alpaca\", split=\"train\")\n",
    "code_dataset = code_dataset.shuffle(seed=42).select(range(min(15000, len(code_dataset))))\n",
    "\n",
    "chat_dataset = load_dataset(\"mlabonne/orpo-dpo-mix-40k\", split=\"train\")\n",
    "chat_dataset = chat_dataset.shuffle(seed=42).select(range(min(6000, len(chat_dataset))))\n",
    "\n",
    "# Format functions (same)\n",
    "def format_code(example):\n",
    "    instruction = example.get('prompt', example.get('instruction', ''))\n",
    "    input_text = example.get('input', '')\n",
    "    output = example.get('completion', example.get('output', ''))\n",
    "    prompt = f\"{instruction}\\n\\nInput: {input_text}\" if input_text else instruction\n",
    "    return f\"[INST] {prompt.strip()} [/INST] {output.strip()}\"\n",
    "\n",
    "def format_chat(example):\n",
    "    prompt = example.get('prompt', example.get('question', example.get('instruction', '')))\n",
    "    response = example.get('chosen', example.get('response', example.get('answer', example.get('output', ''))))\n",
    "    if isinstance(response, list):\n",
    "        response = response[0] if response else \"\"\n",
    "    return f\"[INST] {prompt.strip()} [/INST] {str(response).strip()}\"\n",
    "\n",
    "code_formatted = code_dataset.map(lambda x: {\"text\": format_code(x)}, remove_columns=code_dataset.column_names)\n",
    "chat_formatted = chat_dataset.map(lambda x: {\"text\": format_chat(x)}, remove_columns=chat_dataset.column_names)\n",
    "full_dataset = concatenate_datasets([code_formatted, chat_formatted]).shuffle(seed=42)\n",
    "\n",
    "print(f\"✅ Dataset ready: {len(full_dataset)} examples\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# TRAINING ARGS (same, continues from checkpoint)\n",
    "training_args = TrainingArguments(\n",
    "    output_dir=state['output_dir'],\n",
    "    num_train_epochs=1,\n",
    "    per_device_train_batch_size=1,\n",
    "    gradient_accumulation_steps=8,\n",
    "    optim=\"paged_adamw_8bit\",\n",
    "    learning_rate=2e-4,\n",
    "    max_grad_norm=0.3,\n",
    "    warmup_ratio=0.05,\n",
    "    lr_scheduler_type=\"cosine\",\n",
    "    logging_steps=25,\n",
    "    save_strategy=\"steps\",\n",
    "    save_steps=500,\n",
    "    fp16=True,\n",
    "    gradient_checkpointing=True,\n",
    "    report_to=\"none\",\n",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# RESUME TRAINING!\n",
    "print(\"🚀 RESUMING TRAINING...\")\n",
    "print(f\"   From step: {state['last_step']}\")\n",
    "print(f\"   From epoch: {state['last_epoch']}\")\n",
    "\n",
    "trainer = SFTTrainer(\n",
    "    model=model,\n",
    "    tokenizer=tokenizer,\n",
    "    train_dataset=full_dataset,\n",
    "    max_seq_length=1024,\n",
    "    args=training_args,\n",
    "    dataset_text_field=\"text\",\n",
    ")\n",
    "\n",
    "# KEY: resume_from_checkpoint=True\n",
    "trainer.train(resume_from_checkpoint=CHECKPOINT_DIR)\n",
    "\n",
    "# Save final\n",
    "trainer.save_pretrained(f\"{state['output_dir']}-final\")\n",
    "tokenizer.save_pretrained(f\"{state['output_dir']}-final\")\n",
    "\n",
    "# Update state\n",
    "save_state(\n",
    "    step=trainer.state.global_step,\n",
    "    epoch=trainer.state.epoch,\n",
    "    loss=trainer.state.log_history[-1].get('loss', 'N/A') if trainer.state.log_history else 'N/A',\n",
    "    output_dir=state['output_dir'],\n",
    "    dataset_info=state['dataset_info']\n",
    ")\n",
    "\n",
    "print(\"✅ Training complete! State saved to training_state.json\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}